In [1]:
import mediapipe as mp
import cv2
import numpy as np
import pandas as pd
import math

# Initialize MediaPipe Pose
mp_pose = mp.solutions.pose
pose = mp_pose.Pose()

# 計算旋轉矩陣轉為歐拉角（單位：度）
def rotation_matrix_to_euler_angles(r):
    sy = math.sqrt(r[0, 0]**2 + r[1, 0]**2)

    singular = sy < 1e-6

    if not singular:
        x = math.atan2(r[2, 1], r[2, 2])# Pitch：上下彎腰
        y = math.atan2(-r[2, 0], sy)   # Yaw：左右轉身
        z = math.atan2(r[1, 0], r[0, 0]) # Roll：身體傾斜
    else:
        x = math.atan2(-r[1, 2], r[1, 1])
        y = math.atan2(-r[2, 0], sy)
        z = 0

    return np.degrees(x), np.degrees(y), np.degrees(z)

# Open video file
video_path = "mp4/HW.mp4"  # Replace with your video path
cap = cv2.VideoCapture(video_path)

# List to store angle data
angles_data = []

frame_idx = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Convert the frame to RGB
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # Perform pose detection
    results = pose.process(frame_rgb)

    if results.pose_landmarks:
        landmarks = results.pose_landmarks.landmark

        # 抓三個關鍵點：Nose (0), Left Hip (23), Right Hip (24)
        point_0 = landmarks[mp_pose.PoseLandmark.NOSE]
        point_23 = landmarks[mp_pose.PoseLandmark.LEFT_HIP]
        point_24 = landmarks[mp_pose.PoseLandmark.RIGHT_HIP]

        # Convert landmarks to numpy arrays
        point_0_coords = np.array([point_0.x, point_0.y, point_0.z])
        point_23_coords = np.array([point_23.x, point_23.y, point_23.z])
        point_24_coords = np.array([point_24.x, point_24.y, point_24.z])

        # 計算中點向量
        midpoint_23_24 = (point_23_coords + point_24_coords) / 2

        # Define the vector from Nose (0) to the midpoint
        vector_0_to_midpoint = midpoint_23_24 - point_0_coords

        # Compute the Z-axis as the vector between points 25 and 26
        z_axis = point_24_coords - point_23_coords
        z_axis = z_axis / np.linalg.norm(z_axis)

        # Compute the Y-axis as the cross product of Z-axis and vector_0_to_midpoint
        y_axis = np.cross(z_axis, vector_0_to_midpoint)
        y_axis = y_axis / np.linalg.norm(y_axis)

        # Compute the X-axis as the cross product of Y-axis and Z-axis
        x_axis = np.cross(y_axis, z_axis)
        x_axis = x_axis / np.linalg.norm(x_axis)

        # 建立旋轉矩陣（X, Y, Z）
        rotation_matrix = np.column_stack((x_axis, y_axis, z_axis))

        # Calculate Euler angles from the rotation matrix
        euler_x, euler_y, euler_z = rotation_matrix_to_euler_angles(rotation_matrix)

        # 方向分類
        if euler_y > 0:
            direction = f"Left {abs(euler_y):.2f}°"
        elif euler_y < 0:
            direction = f"Right {abs(euler_y):.2f}°"
        else:
            direction = "Center 0°"

        # Append angles to the list
        angles_data.append({
            "Frame": frame_idx,
            "Euler_X(Pitch)": euler_x,
            "Euler_Y(Yaw)": euler_y,
            "Euler_Z(Roll)": euler_z,
            "Direction": direction
        })

    frame_idx += 1

# Release video capture
cap.release()

# Create DataFrame from the collected data and display it
angles_df = pd.DataFrame(angles_data)
display(angles_df)

,Frame,Euler_X(Pitch),Euler_Y(Yaw),Euler_Z(Roll),Direction
0,0,-26.658944,-46.491354,34.180978,Right 46.49°
1,1,-31.889541,-42.739267,42.577706,Right 42.74°
2,2,-32.429734,-41.784345,44.043324,Right 41.78°
3,3,-30.689905,-42.890520,41.424760,Right 42.89°
4,4,-30.563588,-43.591328,41.015479,Right 43.59°
...,...,...,...,...,...
298,298,-178.741876,-0.290418,103.707102,Right 0.29°
299,299,-178.593420,-0.314883,103.857233,Right 0.31°
300,300,-177.055096,-0.279345,104.481907,Right 0.28°
301,301,-176.330606,-0.152162,104.699715,Right 0.15°
